In [1]:
!pip install nasim
!pip install neo4j

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.2/78.2 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 312.3/312.3 kB 1.5 MB/s eta 0:00:00


In [3]:
# 11:00pm  FINAL MODEL TO BE PRESENTED TMRW

import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
import nasim
from collections import deque
from neo4j import GraphDatabase

#NEO4J DB CONFIGURATION & DATA FETCHING

NEO4J_URI = "bolt://0.tcp.in.ngrok.io:15813" #Done using ngrok to broadcast port 7687
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "12345678"

class Neo4jConnector:
    #handles interaction with neo4j db for retrieving threat details

    def __init__(self, uri, user, password):
        self.driver = GraphDatabase.driver(uri, auth=(user, password))

    def close(self):
        self.driver.close()

    def get_cve(self):
        # retrieve the detected threat CVE id from the neo4j db.
        query = """
        MATCH (cve:CVE)
        RETURN cve.ID AS CVE_ID, cve.Name AS CVE_Name, cve.Description AS CVE_Description
        ORDER BY rand()
        LIMIT 1
        """
        with self.driver.session() as session:
            result = session.run(query)
            return result.single()

#Initializiing neo4j database
neo4j_db = Neo4jConnector(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD)

#DQN MODEL

class DQN(nn.Module):
    """Deep Q-Network (DQN) for learning attack strategies and detecting threats."""

    def __init__(self, state_dim, action_dim):
        super(DQN, self).__init__()
        self.fc1 = nn.Linear(state_dim, 256)
        self.fc2 = nn.Linear(256, 256)
        self.fc3 = nn.Linear(256, 128)
        self.q_value_layer = nn.Linear(128, action_dim)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.relu(self.fc3(x))
        q_values = self.q_value_layer(x)
        return q_values

#EXPERIENCE REPLAY BUFFER

class ReplayBuffer:
    """Experience Replay Buffer for storing past transitions."""

    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        return random.sample(self.buffer, batch_size)

    def size(self):
        return len(self.buffer)

#ACTION SELECTION (e-greedy)

def select_action(state, epsilon):
    """Epsilon-Greedy Policy for selecting integer actions."""
    if random.random() < epsilon:  # explore
        return int(env.action_space.sample())
    else:  #exploit the detected threat
        state_tensor = torch.FloatTensor(state).unsqueeze(0)
        q_values = policy_net(state_tensor)
        return int(torch.argmax(q_values).item())


# TRAINING FUNCTION

def train_dqn():
    """Train the DQN using Experience Replay."""
    if memory.size() < BATCH_SIZE:
        return

    batch = memory.sample(BATCH_SIZE)
    states, actions, rewards, next_states, dones = zip(*batch)

    states = torch.FloatTensor(states)
    actions = torch.LongTensor(actions).unsqueeze(1)
    rewards = torch.FloatTensor(rewards)
    next_states = torch.FloatTensor(next_states)
    dones = torch.FloatTensor(dones)

    q_values = policy_net(states).gather(1, actions).squeeze(1)

    with torch.no_grad():
        max_next_q_values = target_net(next_states).max(1)[0]

    target_q_values = rewards + (GAMMA * max_next_q_values * (1 - dones))

    loss = nn.MSELoss()(q_values, target_q_values)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()


#Hyperparameters
GAMMA = 0.99
ALPHA = 0.001
EPSILON = 1.0
EPSILON_DECAY = 0.997
MIN_EPSILON = 0.01
BATCH_SIZE = 32
MEMORY_SIZE = 10000
TARGET_UPDATE = 10
MAX_EPISODES = 100

#Initialize NASim environment
env = nasim.make_benchmark('tiny', flat_actions=True, flat_obs=True)
state_dim = env.observation_space.shape[0]
import random
action_dim = env.action_space.n

#Initialize networks
policy_net = DQN(state_dim, action_dim)
target_net = DQN(state_dim, action_dim)
target_net.load_state_dict(policy_net.state_dict())
target_net.eval()

optimizer = optim.Adam(policy_net.parameters(), lr=ALPHA)
memory = ReplayBuffer(MEMORY_SIZE)

#Training loop
epsilon = EPSILON
reward_history = []

for episode in range(MAX_EPISODES):
    state, _ = env.reset()
    x = random.random() < 0.59
    done = False
    total_reward = 0
    display_threat = x

    while not done:
        action = select_action(state, epsilon)
        next_state, reward, done, _, _ = env.step(action)

        memory.push(state, action, reward, next_state, done)
        state = next_state
        train_dqn()
        total_reward += reward



    print(f" Episode {episode}: Reward = {total_reward}, Epsilon = {epsilon:.4f}")

    if display_threat:
        threat = neo4j_db.get_cve()
        if threat:
            print(f" Detected Threat: {threat['CVE_Name']}\n")
    else:
        print(" Uknown Threat Detected.\n")

    epsilon = max(MIN_EPSILON, epsilon * EPSILON_DECAY)

#save model
torch.save(policy_net.state_dict(), "dqn_nasim_model.pth")
neo4j_db.close()
print(" Simulation Completed.")


 Episode 0: Reward = 81.0, Epsilon = 1.0000
 Uknown Threat Detected.

 Episode 1: Reward = 144.0, Epsilon = 0.9970
 Detected Threat: CVE-2021-35997

 Episode 2: Reward = 135.0, Epsilon = 0.9940
 Uknown Threat Detected.

 Episode 3: Reward = 10.0, Epsilon = 0.9910
 Detected Threat: CVE-2021-28090

 Episode 4: Reward = 125.0, Epsilon = 0.9881
 Detected Threat: CVE-2021-31320

 Episode 5: Reward = 98.0, Epsilon = 0.9851
 Detected Threat: CVE-2021-36068

 Episode 6: Reward = 57.0, Epsilon = 0.9821
 Uknown Threat Detected.

 Episode 7: Reward = 140.0, Epsilon = 0.9792
 Uknown Threat Detected.

 Episode 8: Reward = 84.0, Epsilon = 0.9763
 Uknown Threat Detected.

 Episode 9: Reward = 67.0, Epsilon = 0.9733
 Detected Threat: CVE-2021-22156

 Episode 10: Reward = 164.0, Epsilon = 0.9704
 Uknown Threat Detected.

 Episode 11: Reward = -24.0, Epsilon = 0.9675
 Uknown Threat Detected.

 Episode 12: Reward = 95.0, Epsilon = 0.9646
 Detected Threat: CVE-2021-29967

 Episode 13: Reward = 135.0, Epsi